# Agentic EEG Decoder — Harmonisasi Data (Tahap 01 → 03)

Notebook **hulu** pipeline: menyiapkan data mentah menjadi artefak siap-training.
Menjalankan **Tahap 01 (akuisisi) → 02 (preprocessing+ICA) → 03 (ekstraksi fitur)**, lalu
mengemas output ke Google Drive sebagai **input untuk [`colab_training.ipynb`](colab_training.ipynb)**.

```
  Zenodo (COG-BCI)  ────unduh────┐
  Drive (STEW.zip) ──ambil zip──┤
                                 ▼
   [01] raw/  →  [02] derivatives/*.fif  →  [03] features/*.npz
                                 │                     │
                    zip→Drive ◄──┘     copy→Drive ◄────┘
                  (derivatives.zip, checkpoint)  (features/)
                                 ▼
                 INPUT untuk colab_training.ipynb
```

## ⚡ Tahan-banting (tahan sesi putus / mati lampu)
Disk Colab `/content` bersifat **ephemeral** — **hilang total** saat sesi mati. Notebook ini
dirancang agar **tidak pernah kehilangan progres**:
1. **Checkpoint ke Drive**: setiap selesai satu *batch* subjek, `derivatives/` di-zip &
   disalin ke `Drive/agentic-eeg/derivatives.zip` (Drive = persisten).
2. **Resume otomatis**: saat dijalankan ulang, notebook **memulihkan** checkpoint dari Drive
   ke `/content`, lalu skrip yang **idempotent** (melewati `*_epo.fif` yang sudah ada)
   langsung melanjutkan dari subjek yang belum selesai.
3. **Unduh resume**: `download_cogbci.py` memakai HTTP Range; **per-batch** sehingga
   kehilangan paling banter = batch yang sedang berjalan.

→ Bila sesi putus: **buka ulang, Runtime → GPU, lalu Run all dari atas.** Aman & lanjut sendiri.

## 🖥️ Runtime: WAJIB GPU
**Runtime → Change runtime type → GPU** (sel 1 meng-`assert` ini). Catatan jujur: langkah
terberat (filter/ICA MNE, PSD, kovarians) **CPU-bound** — GPU tidak mempercepatnya. GPU tetap
diwajibkan karena (a) runtime GPU Colab memberi **VM lebih kuat & sesi lebih panjang**
(mengurangi timeout), (b) ICLabel/onnxruntime memakainya bila tersedia, (c) konsisten dengan
[`colab_training.ipynb`](colab_training.ipynb). **Proteksi timeout sesungguhnya = checkpoint/resume di atas.**

## Hubungan dengan notebook training
| Notebook | Peran | Menghasilkan / memakai |
|---|---|---|
| **colab_harmonisasi.ipynb** (ini) | Menyiapkan data | **Menghasilkan** `features/*.npz` + `derivatives.zip` di Drive |
| [`colab_training.ipynb`](colab_training.ipynb) | Eksperimen Tahap 05–06 | **Memakai** artefak di atas |

## Prasyarat (sekali)
1. **GPU runtime** (lihat di atas).
2. **PAT GitHub** (scope `repo`) via `getpass` — lihat [`README_COLAB.md`](README_COLAB.md) §Langkah-1.
3. **STEW**: unggah manual `STEW_Dataset.zip` ke `MyDrive/agentic-eeg/` (IEEE login-gated).
4. **COG-BCI** diunduh otomatis dari Zenodo (tak perlu Drive).

## 0. Parameter (knob utama)
Atur di sini, sel lain mengikutinya. **Tinjau sebelum Run all.**

In [ ]:
# --- Subjek COG-BCI ---
# Pilot (cepat): ['01','02','03']  |  Penuh 29 subjek: []  (kosong = semua 01..29)
SUBJECTS_COGBCI = ['01', '02', '03']

# --- Tahan-banting / disk ---
BATCH_SIZE = 3                    # subjek COG-BCI per batch; checkpoint ke Drive tiap batch
DELETE_ZIP_AFTER_EXTRACT = True   # hapus sub-*.zip setelah ekstrak (hemat disk)
DELETE_RAW_AFTER_PREPROC = True   # hapus raw subjek setelah preprocess (hemat disk; derivatives sudah aman di Drive)

# --- Path (umumnya tetap) ---
REPO_DIR = '/content/agentic-eeg-decoder'
ORG_REPO = 'AgentiAI-neural-decoding/agentic-eeg-decoder'
DRIVE    = '/content/drive/MyDrive/agentic-eeg'
STEW_ZIP = f'{DRIVE}/STEW_Dataset.zip'
STEW_MD5 = '987330cd36004f1669a2e4dba4f865ba'   # acuan integritas (README_DATASET.md)

ALL_SUBS = SUBJECTS_COGBCI or [f'{i:02d}' for i in range(1, 30)]
print('COG-BCI subjek :', ALL_SUBS if SUBJECTS_COGBCI else f'SEMUA 29 ({ALL_SUBS[0]}..{ALL_SUBS[-1]})')
print('BATCH_SIZE     :', BATCH_SIZE, '| hapus zip:', DELETE_ZIP_AFTER_EXTRACT, '| hapus raw:', DELETE_RAW_AFTER_PREPROC)

## 1. Cek GPU (wajib) + environment
Notebook menolak lanjut bila GPU mati. Cek juga disk & RAM (COG-BCI per subjek ≈ 1,1 GB zip).

In [ ]:
import sys, torch
!nvidia-smi -L
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU mati! Runtime > Change runtime type > GPU, lalu jalankan ulang.'
!echo '--- Disk (/content) ---'; df -h /content | tail -1
!echo '--- RAM ---'; free -h | awk 'NR==1||NR==2'

## 2. Mount Google Drive
Drive = sumber `STEW_Dataset.zip` **dan** tempat checkpoint/hasil persisten.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(f'{DRIVE}/features', exist_ok=True)
print('Isi Drive agentic-eeg:')
!ls -la {DRIVE}

## 3. Clone repo privat (via PAT)
PAT lewat `getpass` — **tak tersimpan**. Bila repo sudah ada (mis. setelah sesi putus), di-`pull`.

In [ ]:
import os, getpass, subprocess
if not os.path.exists(REPO_DIR):
    pat = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone',f'https://{pat}@github.com/{ORG_REPO}.git',REPO_DIR], check=True)
    del pat
else:
    subprocess.run(['git','-C',REPO_DIR,'pull'], check=True)
%cd {REPO_DIR}
!git log --oneline -1

## 4. Install dependensi
Sesuai [`02_Preprocessing/requirements.txt`](02_Preprocessing/requirements.txt) + `pyriemann`
(Tahap 03). **`torch` tak di-reinstall** (pakai CUDA bawaan Colab). `onnxruntime` utk ICLabel.

In [ ]:
!pip install -q mne mne-bids mne-icalabel pyriemann onnxruntime
import mne; print('mne', mne.__version__, '| OK')

## 5. Helper tahan-banting + PULIHKAN checkpoint
Mendefinisikan fungsi orkestrasi & **memulihkan progres** dari `Drive/derivatives.zip` bila
ada (mis. setelah sesi putus). Setelah pulih, skrip idempotent melanjutkan otomatis.

**Checkpoint bersifat atomik**: ditulis ke `*.tmp` di Drive lalu `os.replace` → bila mati saat
menyalin, checkpoint lama tetap utuh.

In [ ]:
import os, glob, shutil, subprocess
PREP = f'{REPO_DIR}/02_Preprocessing'
DERIV = f'{PREP}/derivatives'
CKPT = f'{DRIVE}/derivatives.zip'

def sh(cmd):
    """Jalankan perintah di REPO_DIR; tampilkan & kembalikan exit code."""
    print('>>', cmd)
    return subprocess.run(cmd, shell=True, cwd=REPO_DIR).returncode

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def n_fif():
    return len(glob.glob(f'{DERIV}/**/*.fif', recursive=True))

def checkpoint_derivatives():
    """Zip derivatives -> Drive secara ATOMIK (tmp lalu replace)."""
    if n_fif() == 0:
        print('  (belum ada .fif — lewati checkpoint)'); return
    local = '/content/derivatives.zip'
    if os.path.exists(local):
        os.remove(local)
    subprocess.run('zip -r -q /content/derivatives.zip derivatives/', shell=True, cwd=PREP, check=True)
    tmp = CKPT + '.tmp'
    shutil.copy(local, tmp)
    os.replace(tmp, CKPT)
    print(f'  ✓ checkpoint -> Drive/derivatives.zip ({n_fif()} .fif)')

def restore_checkpoint():
    """Pulihkan derivatives dari checkpoint Drive ke /content (resume)."""
    if os.path.exists(CKPT):
        print('memulihkan derivatives dari checkpoint Drive...')
        os.makedirs(PREP, exist_ok=True)
        os.system(f'unzip -q -o "{CKPT}" -d "{PREP}/"')
        print(f'  ✓ dipulihkan: {n_fif()} .fif')
    else:
        print('tak ada checkpoint sebelumnya — mulai dari awal.')

restore_checkpoint()

## 6. Tahap 01b + 02 — STEW (verifikasi → ekstrak → preprocess → checkpoint)
STEW kecil & cepat, dikerjakan sekali di depan. **Cek MD5** dulu (byte-exact vs manifest),
lalu ekstrak + preprocess, lalu checkpoint. Idempotent: dilewati bila derivatives STEW sudah ada.

In [ ]:
import hashlib, os
assert os.path.exists(STEW_ZIP), f'STEW_Dataset.zip tak ada di {STEW_ZIP} — unggah dulu ke Drive!'
h = hashlib.md5()
with open(STEW_ZIP, 'rb') as f:
    for b in iter(lambda: f.read(1 << 20), b''):
        h.update(b)
got = h.hexdigest()
print('MD5  :', got)
print('acuan:', STEW_MD5, '->', 'OK byte-exact' if got == STEW_MD5 else 'MISMATCH!')
assert got == STEW_MD5, 'MD5 STEW tidak cocok — unduh ulang dari IEEE DataPort.'

In [ ]:
if glob.glob(f'{DERIV}/STEW/sub-*/*.fif'):
    print('derivatives STEW sudah ada — lewati (resume).')
else:
    sh(f'python 01_Akuisisi_Data/scripts/verify_stew.py --zip "{STEW_ZIP}" --out 01_Akuisisi_Data/raw/STEW --keep-zip')
    sh('python 02_Preprocessing/scripts/preprocess_stew.py --config 02_Preprocessing/config/stew.json')
    checkpoint_derivatives()

## 7. Tahap 01a + 02 — COG-BCI (loop per-batch, tahan-banting)
Untuk tiap batch subjek: **unduh (Zenodo, MD5) → ekstrak → preprocess (ICA+ICLabel) →
checkpoint ke Drive → hapus raw**. Batch yang subjeknya sudah ada di derivatives **dilewati**
(resume). Bila sesi putus di tengah, jalankan ulang sel ini — lanjut dari batch berikutnya.

> ⏱️ ICA per rekaman berat. Pilot 3 subjek ≈ puluhan menit; penuh 29 = berjam-jam (terbagi
> aman per batch). Parameter: [`config/cogbci.json`](02_Preprocessing/config/cogbci.json).

In [ ]:
COG_RAW = f'{REPO_DIR}/01_Akuisisi_Data/raw/COG-BCI'

def cog_done(sub):
    return len(glob.glob(f'{DERIV}/COG-BCI/sub-{sub}/ses-*/eeg/*.fif')) > 0

def cog_extract():
    for z in sorted(glob.glob(f'{COG_RAW}/sub-*.zip')):
        os.system(f'unzip -n -q "{z}" -d "{COG_RAW}"')
        if DELETE_ZIP_AFTER_EXTRACT:
            os.remove(z)

for bi, batch in enumerate(chunks(ALL_SUBS, BATCH_SIZE), 1):
    todo = [s for s in batch if not cog_done(s)]
    if not todo:
        print(f'[batch {bi}] {batch} sudah ada -> skip (resume)'); continue
    print(f'\n=== [batch {bi}] memproses {todo} ===')
    sub_flag = ' '.join(todo)
    sh(f'python 01_Akuisisi_Data/scripts/download_cogbci.py --out 01_Akuisisi_Data/raw/COG-BCI --subjects {sub_flag}')
    cog_extract()
    sh(f'python 02_Preprocessing/scripts/preprocess_cogbci.py --config 02_Preprocessing/config/cogbci.json --subjects {sub_flag}')
    checkpoint_derivatives()
    if DELETE_RAW_AFTER_PREPROC:
        for s in todo:
            shutil.rmtree(f'{COG_RAW}/sub-{s}', ignore_errors=True)
        print('  raw batch dihapus (derivatives sudah aman di Drive).')
print('\nCOG-BCI selesai.')

## 7c. Verifikasi derivatives
Hitung `.fif` hasil preprocessing kedua dataset.

In [ ]:
n_cog  = len(glob.glob(f'{DERIV}/COG-BCI/sub-*/ses-*/eeg/*.fif'))
n_stew = len(glob.glob(f'{DERIV}/STEW/sub-*/*.fif'))
print(f'COG-BCI .fif : {n_cog}')
print(f'STEW .fif    : {n_stew}')
assert n_cog > 0 and n_stew > 0, 'derivatives kosong — cek sel 6/7'
print('OK — derivatives siap untuk ekstraksi fitur')

## 8. Tahap 03 — Ekstraksi fitur (harmonisasi cross-dataset)
Harmonisasi 14 kanal bersama + resample **128 Hz** → PSD band-power (70), kovarians (14×14),
tangent global (105). Output `features/*.npz`. Cepat & idempotent (regenerasi penuh tiap jalan).
Parameter: [`config/features.json`](03_Features/config/features.json).

In [ ]:
!python 03_Features/scripts/extract_features.py --config 03_Features/config/features.json

In [ ]:
import glob, json, os
feat = f'{REPO_DIR}/03_Features/features'
npz = sorted(glob.glob(f'{feat}/*.npz'))
print(f'{len(npz)} file .npz:')
for p in npz:
    print('  ', p.split('/')[-1])
man = f'{feat}/feature_manifest.json'
if os.path.exists(man):
    m = json.load(open(man))
    print('\nPSD fitur:', m.get('n_psd_features'), '| tangent:', m.get('n_tangent_features'))
    for k, v in m.get('outputs', {}).items():
        print(f'  {k}: {v["n_epochs"]} epoch, {v["n_subjects"]} subj, kelas {v["class_counts"]}')

## 9. Kemas hasil final ke Drive (input untuk training)
`features/*.npz` di-copy + checkpoint `derivatives.zip` final di-refresh. Keduanya dikonsumsi
[`colab_training.ipynb`](colab_training.ipynb).

In [ ]:
!mkdir -p {DRIVE}/features && cp -v {REPO_DIR}/03_Features/features/*.npz {DRIVE}/features/
checkpoint_derivatives()   # refresh derivatives.zip final di Drive
!echo '--- Isi Drive sekarang ---'; ls -la {DRIVE} {DRIVE}/features

## Selesai ✅ — Langkah berikutnya
Drive `MyDrive/agentic-eeg/` kini berisi `features/*.npz` + `derivatives.zip` →
**buka [`colab_training.ipynb`](colab_training.ipynb)** untuk eksperimen Tahap 05–06.

### Tips & troubleshooting
- **Sesi putus / mati lampu**: buka ulang → Runtime → GPU → **Run all**. Sel 5 memulihkan
  checkpoint Drive, loop melewati batch yang sudah jadi. Tak ada progres hilang.
- **Run penuh 29 subjek**: set `SUBJECTS_COGBCI=[]`. Naikkan `BATCH_SIZE` (mis. 5) untuk
  mengurangi frekuensi re-zip checkpoint (tradeoff: window kehilangan lebih besar bila putus).
- **Disk penuh**: biarkan `DELETE_ZIP_AFTER_EXTRACT` & `DELETE_RAW_AFTER_PREPROC` `True`
  (default) — raw dibuang setelah derivatives aman di Drive.
- **Regenerasi fitur saja** (derivatives sudah di Drive): Run all — sel 5 memulihkan, sel
  6/7 skip, sel 8 mengekstrak ulang fitur.
- **MD5 STEW mismatch**: unduh ulang dari IEEE; jangan lanjut dengan file beda.
- Acuan penuh: COG-BCI `.fif` ≈ 1044, STEW ≈ 86; fitur COG-BCI ≈ 138.900 epoch, STEW ≈ 9.491
  (lihat [`README_DATASET.md`](README_DATASET.md)).